In [7]:
# ==============================
# 📚 RAG SETUP (FULL VERSION)
# ==============================
from langchain_community.document_loaders import PyPDFLoader
from google.colab import files

use_rag = False
pdf_mode = False
rag_index = None
rag_docs = None
embed_model = None

choice = input("Upload PDF for RAG? (yes/no): ").lower()

if choice == "yes":
    use_rag = True
    pdf_mode = True

    uploaded = files.upload()
    pdf_file = list(uploaded.keys())[0]

    loader = PyPDFLoader(pdf_file)
    documents = loader.load()

    texts = [doc.page_content for doc in documents]

    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = embed_model.encode(texts)

    rag_index = faiss.IndexFlatL2(embeddings.shape[1])
    rag_index.add(np.array(embeddings))
    rag_docs = texts

    print("✅ PDF Loaded → RAG ENABLED")
else:
    print("➡️ Running without RAG")


# ==============================
# 🔍 RAG FUNCTIONS
# ==============================
def retrieve_docs(query, k=3):
    q_emb = embed_model.encode([query])
    D, I = rag_index.search(np.array(q_emb), k)
    return [rag_docs[i] for i in I[0]]


def rag_answer(query):
    context = retrieve_docs(query)

    prompt = f"""
    Answer ONLY using the given context.

    Context:
    {context}

    Question:
    {query}
    """

    return llm.invoke(prompt).content


# ==============================
# 🔥 UPDATED MAIN AGENT
# ==============================
def finance_agent(query):
    global pdf_mode

    # =========================
    # 🧠 NORMALIZE QUERY (NEW)
    # =========================
    query = normalize_query(query)
    query_lower = query.lower()

    print(f"🔍 Optimized Query: {query}")  # debug (optional)

    # =========================
    # 📄 PDF MODE
    # =========================
    if pdf_mode and use_rag:
        return rag_answer(query)

    # =========================
    # 🔥 HARD RULE (NOW STRONGER)
    # =========================
    import re
    if re.search(r"(price|trading|stock|share)", query_lower):
        ticker = get_ticker(query) or fallback_ticker_from_llm(query)

        if ticker:
            price = get_price(ticker)
            if price:
                return f"📈 {ticker} Price: ${price:.2f}"

    # =========================
    # 🧠 ROUTING
    # =========================
    route = route_query(query)

    if route == "general":
        return llm.invoke(query).content

    return react_agent(query)

def normalize_query(query):
    prompt = f"""
    You are a query optimizer.

    Fix spelling mistakes, grammar, and make the query clear and concise.
    Do NOT change meaning.

    Examples:
    - "currenly tesla traded at which price?" → "current Tesla stock price"
    - "apl share rate now" → "Apple stock price now"

    Query: {query}

    Return ONLY improved query.
    """

    try:
        improved = llm.invoke(prompt).content.strip()
        return improved
    except:
        return query

    # =========================
    # 2. 🔥 HARD RULE (PRICE FIX)
    # =========================
    if any(k in query_lower for k in ["price", "current", "now", "trading"]):
        ticker = get_ticker(query) or fallback_ticker_from_llm(query)

        if ticker:
            price = get_price(ticker)
            if price:
                return f"📈 {ticker} Price: ${price:.2f}"

    # =========================
    # 3. 🧠 SMART ROUTING
    # =========================
    route = route_query(query)

    # =========================
    # 4. GENERAL
    # =========================
    if route == "general":
        return llm.invoke(query).content

    # =========================
    # 5. 🤖 REACT AGENT
    # =========================
    return react_agent(query)


# ==============================
# 💬 CHAT LOOP (UPDATED)
# ==============================
print("\n🚀 Advanced Finance AI Ready!")
print("Commands:")
print("- exit → quit")
print("- pdf → enter PDF mode")
print("- exitpdf → exit PDF mode\n")

while True:
    q = input("You: ").strip()

    if q.lower() in ["exit", "quit"]:
        print("👋 Goodbye!")
        break

    # ENTER PDF MODE
    if q.lower() == "pdf":
        if use_rag:
            pdf_mode = True
            print("📄 Entered PDF mode\n")
        else:
            print("❌ No PDF uploaded\n")
        continue

    # EXIT PDF MODE
    if q.lower() == "exitpdf":
        pdf_mode = False
        print("📤 Exited PDF mode\n")
        continue

    ans = finance_agent(q)
    print(f"\n🤖 {ans}\n")

Upload PDF for RAG? (yes/no): no
➡️ Running without RAG

🚀 Advanced Finance AI Ready!
Commands:
- exit → quit
- pdf → enter PDF mode
- exitpdf → exit PDF mode

You: currantly tesla stock traded at which price? 
🔍 Optimized Query: Current Tesla stock price

🤖 📈 TSLA Price: $344.23

You: what is marker cap of tesla
🔍 Optimized Query: Tesla market capitalization

🤖 As of my knowledge cutoff in 2023, Tesla's market capitalization was around $830 billion USD. However, please note that market capitalization can fluctuate constantly and may have changed since my knowledge cutoff.

To get the most up-to-date information, I recommend checking a reliable financial website or platform, such as:

* Yahoo Finance
* Google Finance
* Bloomberg
* CNBC

These websites provide real-time data on stock prices, market capitalization, and other financial metrics. You can search for Tesla's stock ticker symbol (TSLA) to get the latest information on its market capitalization.

You: what is stock?
🔍 Optimized

ERROR:yfinance:$NO: possibly delisted; no price data found  (period=1d)



🤖 A stock, also known as equity, is a type of security that represents ownership in a company. When you buy a stock, you essentially buy a small portion of that company's assets and profits.

Here's a simplified explanation:

**What is a stock?**

Imagine a company like a pizza that's cut into small slices. Each slice represents a share of the company, and when you buy a slice (or a share), you become a part-owner of the company. The value of your slice (or share) can fluctuate based on the company's performance and the overall market conditions.

**Key characteristics of stocks:**

1. **Ownership**: As a shareholder, you have a claim on a portion of the company's assets and profits.
2. **Voting rights**: You may have the right to vote on important company decisions, such as electing the board of directors or approving major transactions.
3. **Potential for growth**: The value of your shares can increase if the company performs well, making your investment more valuable.
4. **Risk**: 